In [1]:
import torch 
import json 
import trimesh 
import os 
from torch.utils.data import Dataset, DataLoader 
import numpy as np 
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import numpy as np


## Dataset

In [2]:
class ScanReferDataset(Dataset):
    def __init__(self, scanrefer_data_path, scannet_dir, num_points=40000):
        # Load the ScanRefer JSON
        with open(scanrefer_data_path, 'r') as f:
            self.scanrefer_data = json.load(f)
            
        self.scannet_dir = scannet_dir
        self.num_points = num_points

    def __len__(self):
        return len(self.scanrefer_data)

    def __getitem__(self, idx):
        item = self.scanrefer_data[idx]
        scene_id = item["scene_id"]
        target_obj_id = str(item["object_id"]) 
        
        # 1. Load the 3D Point Cloud (.ply)
        ply_path = os.path.join(self.scannet_dir, "scans", scene_id, f"{scene_id}_vh_clean_2.ply")
        mesh = trimesh.load(ply_path, process=False)
        
        points = np.array(mesh.vertices) 
        colors = np.array(mesh.visual.vertex_colors[:, :3]) / 255.0 
        
        # 2. Extract Ground Truth from JSONs using raw points
        agg_path = os.path.join(self.scannet_dir, "scans", scene_id, f"{scene_id}.aggregation.json")
        with open(agg_path, 'r') as f:
            agg_data = json.load(f)
            
        target_segments = []
        for seg_group in agg_data['segGroups']:
            if str(seg_group['objectId']) == target_obj_id:
                target_segments = seg_group['segments']
                break
                
        segs_path = os.path.join(self.scannet_dir, "scans", scene_id, f"{scene_id}_vh_clean_2.0.010000.segs.json")
        with open(segs_path, 'r') as f:
            segs_data = json.load(f)
            
        seg_indices = np.array(segs_data['segIndices'])
        valid_vertex_mask = np.isin(seg_indices, target_segments)
        target_points = points[valid_vertex_mask]
        
        # Calculate the Bounding Box
        if len(target_points) > 0:
            gt_min = np.min(target_points, axis=0)
            gt_max = np.max(target_points, axis=0)
            gt_box_center = (gt_max + gt_min) / 2.0
            gt_box_size = gt_max - gt_min
        else:
            gt_box_center = np.zeros(3)
            gt_box_size = np.ones(3) * 1e-6 
            
        # 3. Downsample the Point Cloud
        point_cloud = np.concatenate([points, colors], axis=1)
        
        if point_cloud.shape[0] > self.num_points:
            choices = np.random.choice(point_cloud.shape[0], self.num_points, replace=False)
            point_cloud = point_cloud[choices, :]
        else:
            padding = np.zeros((self.num_points - point_cloud.shape[0], 6))
            point_cloud = np.vstack((point_cloud, padding))
            
        text_tokens = item["token"]
        raw_text = " ".join(text_tokens)
        
        return {
            "point_cloud": torch.tensor(point_cloud, dtype=torch.float32),
            "gt_box_center": torch.tensor(gt_box_center, dtype=torch.float32),
            "gt_box_size": torch.tensor(gt_box_size, dtype=torch.float32),
            "text": raw_text
        }

    def visualize_point_cloud(self, idx):
        
        datapoint = self.__getitem__(idx)
        
        # Convert tensors back to numpy arrays for plotting
        point_cloud = datapoint["point_cloud"].numpy()
        gt_box_center = datapoint["gt_box_center"].numpy()
        gt_box_size = datapoint["gt_box_size"].numpy()
        text = datapoint["text"]
        
        # Extract coordinates and colors
        points = point_cloud[:, :3]
        colors = point_cloud[:, 3:6]
        
        # Filter out the padding (rows that are entirely zero)
        mask = np.any(points != 0, axis=1)
        points = points[mask]
        colors = colors[mask]
        
        # Create a 3D plot
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        # Plot the point cloud
        # Matplotlib can be sluggish with 40k points. Slicing [::2] or [::4] 
        # subsamples the points for smoother rendering if needed.
        ax.scatter(points[::2, 0], points[::2, 1], points[::2, 2], 
                   c=colors[::2], s=0.5, marker='.')
        
        # Calculate bounding box corners
        cx, cy, cz = gt_box_center
        lx, ly, lz = gt_box_size
        
        x_min, x_max = cx - lx/2.0, cx + lx/2.0
        y_min, y_max = cy - ly/2.0, cy + ly/2.0
        z_min, z_max = cz - lz/2.0, cz + lz/2.0
        
        # Define the 8 corners of the bounding box
        corners = np.array([
            [x_min, y_min, z_min], [x_max, y_min, z_min],
            [x_max, y_max, z_min], [x_min, y_max, z_min],
            [x_min, y_min, z_max], [x_max, y_min, z_max],
            [x_max, y_max, z_max], [x_min, y_max, z_max]
        ])
        
        # Define the 12 edges connecting the corners
        edges = [
            [0, 1], [1, 2], [2, 3], [3, 0], # Bottom face
            [4, 5], [5, 6], [6, 7], [7, 4], # Top face
            [0, 4], [1, 5], [2, 6], [3, 7]  # Vertical pillars
        ]
        
        # Plot the bounding box edges
        for edge in edges:
            ax.plot(corners[edge, 0], corners[edge, 1], corners[edge, 2], 
                    color='red', linewidth=2.5)
            
        # Set axis limits to maintain a 1:1:1 aspect ratio (so the box doesn't distort)
        max_range = np.array([points[:,0].max()-points[:,0].min(), 
                              points[:,1].max()-points[:,1].min(), 
                              points[:,2].max()-points[:,2].min()]).max() / 2.0
        
        mid_x = (points[:,0].max() + points[:,0].min()) * 0.5
        mid_y = (points[:,1].max() + points[:,1].min()) * 0.5
        mid_z = (points[:,2].max() + points[:,2].min()) * 0.5
        
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
        
        # Set labels and title
        ax.set_title(f"Target: '{text}'\nBox Center: {np.round(gt_box_center, 2)}")
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        
        plt.show()

    def visualize_point_cloud_interactive(self, idx):
        import plotly.graph_objects as go
        import numpy as np
        
        datapoint = self.__getitem__(idx)
        
        # Convert tensors back to numpy arrays
        point_cloud = datapoint["point_cloud"].numpy()
        gt_box_center = datapoint["gt_box_center"].numpy()
        gt_box_size = datapoint["gt_box_size"].numpy()
        text = datapoint["text"]
        
        # Extract coordinates and colors, filter padding
        points = point_cloud[:, :3]
        colors = point_cloud[:, 3:6]
        
        mask = np.any(points != 0, axis=1)
        points = points[mask]
        colors = colors[mask]
        
        # Subsample for even smoother rendering (optional)
        step = 2
        p_sub = points[::step]
        c_sub = colors[::step]
        
        # Plotly expects colors as an array of RGB strings
        color_strings = [f'rgb({int(r*255)}, {int(g*255)}, {int(b*255)})' 
                         for r, g, b in c_sub]
        
        # 1. Create the Point Cloud Trace
        trace_pc = go.Scatter3d(
            x=p_sub[:, 0], y=p_sub[:, 1], z=p_sub[:, 2],
            mode='markers',
            marker=dict(
                size=1.5,
                color=color_strings,
                opacity=0.8
            ),
            name='Point Cloud'
        )
        
        # 2. Calculate Bounding Box Corners
        cx, cy, cz = gt_box_center
        lx, ly, lz = gt_box_size
        
        x_min, x_max = cx - lx/2.0, cx + lx/2.0
        y_min, y_max = cy - ly/2.0, cy + ly/2.0
        z_min, z_max = cz - lz/2.0, cz + lz/2.0
        
        corners = np.array([
            [x_min, y_min, z_min], [x_max, y_min, z_min],
            [x_max, y_max, z_min], [x_min, y_max, z_min],
            [x_min, y_min, z_max], [x_max, y_min, z_max],
            [x_max, y_max, z_max], [x_min, y_max, z_max]
        ])
        
        # Plotly connects lines sequentially. We use None to break the line.
        edges = [
            [0, 1, 2, 3, 0], # Bottom face
            [4, 5, 6, 7, 4], # Top face
            [0, 4], [1, 5], [2, 6], [3, 7] # Pillars
        ]
        
        box_x, box_y, box_z = [], [], []
        for edge_seq in edges:
            for idx in edge_seq:
                box_x.append(corners[idx, 0])
                box_y.append(corners[idx, 1])
                box_z.append(corners[idx, 2])
            box_x.append(None) # Break line
            box_y.append(None)
            box_z.append(None)
            
        # 3. Create the Bounding Box Trace
        trace_box = go.Scatter3d(
            x=box_x, y=box_y, z=box_z,
            mode='lines',
            line=dict(color='red', width=4),
            name='Bounding Box'
        )
        
        # 4. Assemble and display the figure
        fig = go.Figure(data=[trace_pc, trace_box])
        
        fig.update_layout(
            title=f"Target: '{text}'<br>Box Center: {np.round(gt_box_center, 2)}",
            scene=dict(
                xaxis_title='X',
                yaxis_title='Y',
                zaxis_title='Z',
                aspectmode='data' # This forces the 1:1:1 scale automatically!
            ),
            margin=dict(l=0, r=0, b=0, t=40)
        )
        
        fig.show()
        


In [3]:
train_scanrefer_dataset = ScanReferDataset(
    scanrefer_data_path="/home/avishka/sasika/grounding/3d/new_data/scanrefer/ScanRefer_filtered_train.json",
    scannet_dir="/home/avishka/sasika/grounding/3d/new_data/data/scannet",
    num_points=40000
)

val_scanrefer_dataset = ScanReferDataset(
    scanrefer_data_path="/home/avishka/sasika/grounding/3d/new_data/scanrefer/ScanRefer_filtered_val.json",
    scannet_dir="/home/avishka/sasika/grounding/3d/new_data/data/scannet",
    num_points=40000
)

In [4]:
train_scanrefer_dataset[0]

{'point_cloud': tensor([[6.0757, 8.2759, 1.3731, 0.5569, 0.5686, 0.5333],
         [1.5695, 6.8182, 1.4450, 0.5412, 0.4902, 0.3373],
         [1.8124, 4.9734, 0.0358, 0.8235, 0.7451, 0.6431],
         ...,
         [6.6771, 1.6628, 1.7196, 0.6118, 0.5373, 0.3961],
         [1.2453, 3.5889, 0.0503, 0.7490, 0.6824, 0.5843],
         [3.1063, 7.4879, 2.5968, 0.3843, 0.4314, 0.4706]]),
 'gt_box_center': tensor([2.1086, 0.7030, 1.0045]),
 'gt_box_size': tensor([0.8171, 1.2794, 1.9381]),
 'text': 'a white cabinet in the corner of the room . in the direction from the door and from the inside . it will be on the left , there is a small brown table on the left side of the cabinet and a smaller table on the right side of the cabinet'}

In [5]:
# train_scanrefer_dataset.visualize_point_cloud_interactive(0)

In [6]:
from transformers import AutoTokenizer, AutoModel

# The tokenizer is perfect
text_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

# Use AutoModel to get the raw hidden states (features)
text_encoder = AutoModel.from_pretrained("FacebookAI/roberta-base")

/home/avishka/anaconda3/envs/sonata/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
